![EPFL Center for Imaging logo](https://imaging.epfl.ch/resources/logo-for-gitlab.svg)
# Image segmentation with the `transformers` library

This notebook was mostly adapted from the official docs: [Image Segmentation](https://huggingface.co/docs/transformers/en/tasks/semantic_segmentation).

We recommend that you take a look at the notebook of introduction *[Fine-Tuning Machine Learning Models with the `transformers` Library](https://github.com/EPFL-Center-for-Imaging/workshop-transformers/blob/main/intro_model_finetuning.ipynb)* before following this notebook to familiarize yourself with model fine-tuning using `transformers`.

**Challenge**

We'll fine-tune a [SegFormer](https://arxiv.org/abs/2105.15203) model ([`nvidia/mit-b0`](https://huggingface.co/nvidia/mit-b0)) to segment out defects in the [MVTec Bottle](https://www.kaggle.com/datasets/ipythonx/mvtec-ad/data) dataset. The training dataset, which contains about 70 images with ground truth mask annotations, can be downloaded from this [link](https://moodle.epfl.ch/).

![test-example](./test-set-bottle.png)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict, Image
from pathlib import Path

Load a local dataset (images and annotated masks):

Here, for simplicity we do the training-validation-test split on the fly (randomly), so we start from a local folder with the structure:

```
bottle
|---- train
    |---- images
          |---- *.png
    |---- annotations
          |---- *_mask.png
```

In [ ]:
dataset_path = "./datasets/bottle"  # Modify if needed

In [ ]:
def create_dataset(image_paths, label_paths):
    dataset = Dataset.from_dict(
        {
            "image": sorted(image_paths),
            "annotation": sorted(label_paths)
        }
    )
    dataset = dataset.cast_column("image", Image())
    dataset = dataset.cast_column("annotation", Image())
    return dataset

train_images_path = Path(dataset_path) / "train" / "images"
train_annotations_path = Path(dataset_path) / "train" / "annotations"

image_paths_train = list(train_images_path.glob("*.png"))
label_paths_train = list(train_annotations_path.glob("*.png"))
image_paths_train = [str(i) for i in image_paths_train]
label_paths_train = [str(i) for i in label_paths_train]

train_dataset = create_dataset(image_paths_train, label_paths_train)

dataset = DatasetDict({"train": train_dataset})

# We do the train/val/test splitting on the fly:
dataset = dataset["train"].train_test_split(test_size=0.2)
test_val_split = dataset["test"].train_test_split(test_size=0.5)
dataset = DatasetDict({
    "train": dataset["train"],
    "validation": test_val_split["train"],
    "test": test_val_split["test"],
})

dataset

Label mappings:

In [ ]:
id2label = {0: 'background', 1: 'defect'}
label2id = {val: key for key, val in id2label.items()}
num_labels = len(id2label)

id2label

Choose a model checkpoint from the [HuggingFace Hub (Models)](https://huggingface.co/models?pipeline_tag=image-segmentation&library=transformers&sort=trending), for example:

In [ ]:
model_id = "nvidia/mit-b0"

Load the image preprocessor:

In [ ]:
from transformers import AutoImageProcessor

image_processor = AutoImageProcessor.from_pretrained(model_id, do_reduce_labels=False)

Prepare transforms, including a "color jitter" augmentation:

In [ ]:
from torchvision.transforms import ColorJitter

jitter = ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.1)

def preprocess_train(batch):
    images = [jitter(x) for x in batch["image"]]
    labels = [x for x in batch["annotation"]]
    inputs = image_processor(images, labels)
    batch["pixel_values"] = inputs["pixel_values"]
    batch["labels"] = inputs["labels"]
    return batch

def preprocess_val(batch):
    images = [x for x in batch["image"]]
    labels = [x for x in batch["annotation"]]
    inputs = image_processor(images, labels)
    batch["pixel_values"] = inputs["pixel_values"]
    batch["labels"] = inputs["labels"]
    return batch

train_ds = dataset["train"].with_transform(preprocess_train)
test_ds = dataset["validation"].with_transform(preprocess_val)

print(dataset["train"][0].keys())
print(train_ds[0].keys())

Load the pretrained model:

In [ ]:
from transformers import AutoModelForSemanticSegmentation

model = AutoModelForSemanticSegmentation.from_pretrained(
    model_id, 
    num_labels=num_labels,
    id2label=id2label, 
    label2id=label2id,
)

Define a data collator function:

In [ ]:
def collate_fn(batch):
    return {
        "pixel_values": torch.stack([item["pixel_values"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }

Compute the mean intersection over union (IoU) as a metric to monitor during training:

In [ ]:
!pip install evaluate

In [ ]:
from torch import nn
import evaluate

metric = evaluate.load("mean_iou")

def compute_metrics(eval_pred):
    # Adapted from: https://huggingface.co/docs/transformers/en/tasks/semantic_segmentation
    with torch.no_grad():
        logits, labels = eval_pred
        logits_tensor = torch.from_numpy(logits)
        logits_tensor = nn.functional.interpolate(
            logits_tensor,
            size=labels.shape[-2:],
            mode="bilinear",
            align_corners=False,
        ).argmax(dim=1)

        pred_labels = logits_tensor.detach().cpu().numpy()
        metrics = metric.compute(
            predictions=pred_labels,
            references=labels,
            num_labels=num_labels,
            ignore_index=255,
            reduce_labels=False,
        )
        for key, value in metrics.items():
            if isinstance(value, np.ndarray):
                metrics[key] = value.tolist()
        
        return metrics

Choose a path to a directory where to save the fine-tuned model:

In [ ]:
finetuned_model_path = f"models/{model_id.split('/')[1]}"

finetuned_model_path

Prepare the `Trainer`:

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=finetuned_model_path,
    eval_strategy="epoch",
    save_strategy="best",
    metric_for_best_model="mean_iou",
    num_train_epochs=10,
    learning_rate=6e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    logging_steps=10,
    load_best_model_at_end=True,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

print(f"Ready to train. Device: {trainer.args.device}")

Train the model:

In [ ]:
trainer.train()

Save the model:

In [ ]:
trainer.save_model(finetuned_model_path)
image_processor.save_pretrained(finetuned_model_path)

## Inference

We test our model on the test set:

In [ ]:
x_test = dataset["test"]["image"]
y_test = dataset["test"]["annotation"]

As usual, we use `pipeline` for inference:

In [ ]:
from transformers import pipeline

segmentor = pipeline("image-segmentation", finetuned_model_path)

results = segmentor(list(x_test))

results[:3]  # First 3 results

Let's plot a few segmentaiton results:

In [ ]:
from PIL.ImageOps import invert

def plot_segmentation_result(idx, image, mask_pred, mask_true):
    fig, axes = plt.subplots(figsize=(15, 5), ncols=3)
    axes[0].imshow(image)
    axes[0].set_title(f'Test image (#{idx})')
    axes[0].set_axis_off()
    axes[1].imshow(mask_pred, interpolation="none")
    axes[1].set_title('Predicted mask')
    axes[1].set_axis_off()
    axes[2].imshow(mask_true, interpolation="none")
    axes[2].set_title('Ground truth mask')
    axes[2].set_axis_off()
    plt.show()

y_pred = [r[0]['mask'] for r in results]

for idx, (image, mask_pred, mask_true) in enumerate(zip(x_test, y_pred, y_test)):
    plot_segmentation_result(idx, image, invert(mask_pred), mask_true)